In [1]:
#导入环境变量
from dotenv import load_dotenv
from langchain_core.messages import AIMessage

load_dotenv()

True

In [2]:
#创建模型
from langchain.chat_models import init_chat_model
import os
load_dotenv(dotenv_path=r"D:\SystemData\Python\damoxing\.env", override=True)

model=init_chat_model(
    model='qwen3.5-omni-plus',
    model_provider='openai',
    base_url=os.getenv('dashscope_base_url'),
    api_key=os.getenv('dashscope_api_key'),
)

In [3]:
#导入json格式
import json

In [4]:
#导入记忆依赖
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

In [5]:
#连接sqlite
connection=sqlite3.connect('对话记忆/checkpoint.db',check_same_thread=False)

In [6]:
#初始化记忆点
checkpointer=SqliteSaver(connection)

In [7]:
#自动建表
checkpointer.setup()

In [8]:
#创建智能体
from langchain.agents import create_agent
agent=create_agent(
    model=model,
    checkpointer=checkpointer,
)

In [9]:
#定一个会话名称
config = {"configurable": {"thread_id": "user-001"}}

In [15]:
from langchain.messages import HumanMessage,AIMessage
response=agent.invoke(
    {'messages':[HumanMessage('你好')]},
    config
)

messages = response.get("messages", []) if isinstance(response, dict) else getattr(response, "messages", [])

print(json.dumps({
    "final_answer": next((m.content for m in reversed(messages) if isinstance(m, AIMessage)), ""),
    "messages": [
        {"role": "human" if isinstance(m, HumanMessage) else "ai", "content": m.content}
        for m in messages
    ]
}, indent=2, ensure_ascii=False))

{
  "final_answer": "你好！很高兴再次见到你。今天有什么想聊的，或者需要我帮忙解决的问题吗？",
  "messages": [
    {
      "role": "human",
      "content": "你好"
    },
    {
      "role": "ai",
      "content": "你好！有什么我可以帮你的吗？"
    },
    {
      "role": "human",
      "content": "你好"
    },
    {
      "role": "ai",
      "content": "你好！很高兴再次见到你。今天有什么想聊的，或者需要我帮忙解决的问题吗？"
    }
  ]
}
